# Backend experiment: aegra

[`aegra`](https://github.com/aegra/aegra) is a self-hosted, open-source alternative to LangGraph Platform (FastAPI + Postgres + Redis), run here via Docker.

This notebook is a self-contained, reproducible experiment: it starts the `aegra` backend, deploys the three example agents (`researcher`, `coder`, `reviewer`), calls each one directly through `langgraph_sdk`, then runs the `supervisor` agent locally and has it orchestrate all three **via `RemoteGraph`**.


In [1]:
import sys

sys.path.insert(0, "/Users/jyje/repo/jyje/pilot-langchain-remotegraph")

from langgraph_sdk import get_sync_client

from remotegraph.backends.aegra import AegraBackend as Backend

backend = Backend()
print(backend.name, "->", backend.base_url)


aegra -> http://127.0.0.1:2026


## 1. Start the backend

`backend.up()` runs `docker compose up -d --build` (postgres + redis + the aegra app). First run builds the image, so this can take a minute or two.

In [2]:
SERVED_AGENTS = {
    "researcher": "agents/researcher/graph.py:graph",
    "coder": "agents/coder/graph.py:graph",
    "reviewer": "agents/reviewer/graph.py:graph",
}
backend.deploy(SERVED_AGENTS)
backend.up()
print(backend.status())


 Image aegra-aegra Building 


#1 [internal] load local bake definitions
#1 reading from stdin 555B done
#1 DONE 0.0s



#2 [internal] load build definition from Dockerfile
#2 transferring dockerfile: 701B done
#2 DONE 0.0s

#3 [auth] library/python:pull token for registry-1.docker.io
#3 DONE 0.0s

#4 [internal] load metadata for docker.io/library/python:3.12-slim-bookworm


#4 DONE 1.2s

#5 [internal] load .dockerignore
#5 transferring context: 2B done
#5 DONE 0.0s

#6 [1/6] FROM docker.io/library/python:3.12-slim-bookworm@sha256:76d4b7b6305788c6b4c6a19d6a22a3921bf802e9af4d5e1e5bd771208dba74bf
#6 DONE 0.0s

#7 [internal] load build context
#7 transferring context: 1.79kB done
#7 DONE 0.0s

#8 [3/6] RUN apt-get update && apt-get install -y --no-install-recommends     build-essential libpq-dev curl     && rm -rf /var/lib/apt/lists/*
#8 CACHED

#9 [4/6] RUN pip install --no-cache-dir     "aegra-cli>=0.9.22"     "langchain>=1.3.10"     "langchain-openai>=1.3.2"     "langgraph>=1.2.6"     "requests>=2.34.2"     "python-dotenv>=1.2.2"
#9 CACHED

#10 [5/6] COPY agents/ ./agents/
#10 CACHED

#11 [2/6] WORKDIR /app
#11 CACHED

#12 [6/6] COPY docker/aegra/aegra.json ./aegra.json
#12 CACHED

#13 exporting to image
#13 exporting layers done
#13 writing image sha256:5611198699d69b3343cad27ac29b0402d7c82090e303cd814b55fcbcd2f995b0 0.0s done
#13 naming to docker.io/libr

#13 naming to docker.io/library/aegra-aegra done
#13 DONE 0.0s

#14 resolving provenance for metadata file
#14 DONE 0.0s


 Image aegra-aegra Built 
 Network aegra_default Creating 
 Network aegra_default Created 
 Container aegra-postgres-1 Creating 
 Container aegra-redis-1 Creating 


 Container aegra-redis-1 Created 
 Container aegra-postgres-1 Created 
 Container aegra-aegra-1 Creating 
 Container aegra-aegra-1 Created 
 Container aegra-postgres-1 Starting 
 Container aegra-redis-1 Starting 


 Container aegra-postgres-1 Started 
 Container aegra-redis-1 Started 
 Container aegra-redis-1 Waiting 
 Container aegra-postgres-1 Waiting 


 Container aegra-redis-1 Healthy 
 Container aegra-postgres-1 Healthy 
 Container aegra-aegra-1 Starting 
 Container aegra-aegra-1 Started 


NAME               IMAGE                    COMMAND                  SERVICE    CREATED         STATUS                                     PORTS
aegra-aegra-1      aegra-aegra              "aegra serve"            aegra      6 seconds ago   Up Less than a second (health: starting)   0.0.0.0:2026->2026/tcp, [::]:2026->2026/tcp
aegra-postgres-1   pgvector/pgvector:pg18   "docker-entrypoint.s…"   postgres   6 seconds ago   Up 6 seconds (healthy)                     0.0.0.0:5432->5432/tcp, [::]:5432->5432/tcp
aegra-redis-1      redis:7-alpine           "docker-entrypoint.s…"   redis      6 seconds ago   Up 6 seconds (healthy)                     0.0.0.0:6379->6379/tcp, [::]:6379->6379/tcp


## 2. List registered assistants

In [3]:
import time

client = get_sync_client(url=backend.base_url)

for _attempt in range(10):
    try:
        assistants = list(client.assistants.search())
        if assistants:
            break
    except Exception as exc:
        print("waiting for backend...", exc)
    time.sleep(3)

for a in assistants:
    print(a["assistant_id"], "graph_id=" + str(a.get("graph_id")))


waiting for backend... [Errno 54] Connection reset by peer


waiting for backend... [Errno 54] Connection reset by peer


ad9aa870-1e45-562b-a83d-73b96694ea13 graph_id=coder
c926ac5a-b04e-5949-878a-8e4830d4338b graph_id=researcher
ca7018db-539a-5a5f-b9c2-8622330bec7d graph_id=reviewer


## 3. Call each agent directly

In [4]:
def call(name: str, message: str) -> str:
    last = None
    stream = client.runs.stream(
        None, name, input={"messages": [{"role": "user", "content": message}]}, stream_mode="values"
    )
    for chunk in stream:
        if chunk.event == "values" and chunk.data.get("messages"):
            last = chunk.data["messages"][-1]["content"]
    return last


print(call("coder", "What is 13 * 7? Just the number."))


91


In [5]:
print(call("researcher", "What is LangGraph in one sentence?"))


LangGraph is a library built by the LangChain team that allows developers to create complex, stateful AI applications—including single or multi-agent systems—by modeling the interaction flow as a deterministic graph.


In [6]:
print(call("reviewer", "Review this code: def add(a, b): return a + b"))


The code is correct and achieves its intended purpose of adding two numbers.

**Verdict:** Passes linting.


## 4. Run the supervisor pipeline via RemoteGraph

The supervisor never imports `researcher`/`coder`/`reviewer` directly -- it only knows the backend's base URL and calls them through `langgraph.pregel.remote.RemoteGraph`, exactly as it would for a real LangGraph Platform deployment.

In [7]:
import os

os.environ["REMOTEGRAPH_BASE_URL"] = backend.base_url

sys.path.insert(0, "/Users/jyje/repo/jyje/pilot-langchain-remotegraph")
from agents.supervisor.graph import graph as supervisor_graph

task = "Write a one-line python function that returns the square of a number."
result = supervisor_graph.invoke({"task": task})
print("--- research ---")
print(result["research"])
print()
print("--- code ---")
print(result["code"])
print()
print("--- review ---")
print(result["review"])


--- research ---
```python
square = lambda x: x * x
```

**(Alternatively, using `pow`):**

```python
square = lambda x: pow(x, 2)
```

--- code ---
```python
square = lambda x: x * x
```

--- review ---
The code is correct and follows good Python style for a simple anonymous function assignment.

**Verdict:** Correct and idiomatic.


## 5. Tear down

In [8]:
backend.down()
print(backend.status())


 Container aegra-aegra-1 Stopping 


 Container aegra-aegra-1 Stopped 
 Container aegra-aegra-1 Removing 
 Container aegra-aegra-1 Removed 
 Container aegra-redis-1 Stopping 
 Container aegra-postgres-1 Stopping 


 Container aegra-redis-1 Stopped 
 Container aegra-redis-1 Removing 
 Container aegra-redis-1 Removed 


NAME      IMAGE     COMMAND   SERVICE   CREATED   STATUS    PORTS


 Container aegra-postgres-1 Stopped 
 Container aegra-postgres-1 Removing 
 Container aegra-postgres-1 Removed 
 Network aegra_default Removing 
 Network aegra_default Removed 
